In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ipynb.fs.full.preprocessing as pp


In [7]:
x_px = 11 #0.1mm
y_px = 11 #0.1mm
x_conversion = 0.1/x_px #px to mm conversion 
y_conversion = 0.1/y_px #px to mm conversion 
data = pp.combine_the_data_1([("0007_mesh", True),("0008_dmesh", True),("0009_dmesh", True),("0010_dmesh", True),("0011_dmesh", True)], "scans/data", x_conversion=x_conversion, y_conversion=y_conversion)
#['', '', '', '', '', '0012_dmesh', '0013_dmesh', '0014_dmesh', '0015_dmesh', '0016_dmesh', '0017_mesh', '0018_mesh']

display(data[["x", "cx_mm", "y", "cy_mm"]].head())


IndexError: index 0 is out of bounds for axis 0 with size 0

For this test example - I am going to estimate the mm to pixels - this estimation has been done separately to the scan so is invalid and it only for testing and logic purposes. 

There is a factor of ~2 different here in the x to cx_mm conversion - probably due to my conversion - the microscope may have been further away when I did the calibration. 

This is just a test to get used to the NN side of things so I will leave it for now.

In [ ]:
plt.scatter(data["cx_mm"], data["cy_mm"])

In [ ]:
import seaborn as sns
sns.pairplot(data[[ "s1", "s2", "cx_mm", "cy_mm"]], diag_kind='kde')

<h1> Tensor Flow Basic Regression Tutorial but with my data </h1>

In [ ]:
#need multiple inputs - s0,s1,s1 and multiple outputs cx_mm, cy_mm
#For now just doing one output - s1 and two inputs - cx_mm, cy_mm (initially just cx_mm)

inputs = data[["cx_mm", "cy_mm"]]
outputs = data[["s0", "s1", "s2"]]
# dataset = data[["cx_mm", "cy_mm", "s0", "s1", "s2"]]
dataset = data[["cx_mm", "cy_mm", "s1"]]
dataset = dataset.reset_index(drop=True) #reset indexes as multiple datapoints combined
# x_train, x_test, y_train, y_test = train_test_split(inputs, outputs, test_size=0.2)
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)
# print(train_dataset)
print(test_dataset)

In [ ]:
train_dataset.tail()

The tutorial is only to predict one output. As s0=0 and s1=-s2,just predict s1 for now 

In [ ]:
train_features = train_dataset.copy()
test_features = test_dataset.copy()

train_labels = train_features.pop('s1')
test_labels = test_features.pop('s1')

print(test_features)
print(test_labels)

In [ ]:
train_dataset.describe().transpose()[['mean', 'std']]


In [ ]:
#normalise features - maybe don't need as all inputs have the same range
import tensorflow as tf
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(np.array(train_features))
print(normalizer.mean.numpy())


In [ ]:
first = np.array(train_features[:1])
print(train_features[:1])
with np.printoptions(precision=2, suppress=True):
  print('First example:', first)
  print()
  print('Normalized:', normalizer(first).numpy())


<h3>Linear Regression

In [ ]:
from tensorflow.keras import layers

cx = np.array(train_features['cx_mm'])

cx_normalizer = layers.Normalization(input_shape=[1,], axis=None)
cx_normalizer.adapt(cx)


In [ ]:
cx_model = tf.keras.Sequential([
    cx_normalizer,
    layers.Dense(units=1)
])

cx_model.summary()


In [ ]:
cx_model.predict(cx[:10])


In [ ]:
cx_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
    loss='mean_absolute_error')


In [ ]:
history = cx_model.fit(
    train_features['cx_mm'],
    train_labels,
    epochs=100,
    # Suppress logging.
    verbose=0,
    # Calculate validation results on 20% of the training data.
    validation_split = 0.2)

In [ ]:
hist = pd.DataFrame(history.history)
hist['epoch'] = history.epoch
hist.tail()

In [ ]:
def plot_loss(history):
  plt.plot(history.history['loss'], label='loss')
  plt.plot(history.history['val_loss'], label='val_loss')
  plt.ylim([0, 1100])
  plt.xlabel('Epoch')
  plt.ylabel('Error [MPG]')
  plt.legend()
  plt.grid(True)
plot_loss(history)


In [ ]:
test_results = {}

test_results['cx_model'] = cx_model.evaluate(
    test_features['cx_mm'],
    test_labels, verbose=0)


In [ ]:
x = tf.linspace(-1.5, 1.5, 251)
y = cx_model.predict(x)


In [ ]:
def plot_cx(x, y):
  plt.scatter(train_features['cx_mm'], train_labels, label='Data')
  plt.plot(x, y, color='k', label='Predictions')
  plt.xlabel('cx_mm')
  plt.ylabel('s1')
  plt.legend()
plot_cx(x, y)


<h4>Multiple Inputs

In [ ]:
#multiple inputs cx_mm and cy_mm
linear_model = tf.keras.Sequential([
    normalizer,
    layers.Dense(units=1)
])
linear_model.predict(train_features[:10])
linear_model.layers[1].kernel
linear_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
    loss='mean_absolute_error')



In [ ]:
history = linear_model.fit(
    train_features,
    train_labels,
    epochs=100,
    # Suppress logging.
    verbose=0,
    # Calculate validation results on 20% of the training data.
    validation_split = 0.2)
plot_loss(history)

In [ ]:
test_results['linear_model'] = linear_model.evaluate(
    test_features, test_labels, verbose=0)


<h3>DNN

In [ ]:
def build_and_compile_model(norm):
  model = tf.keras.Sequential([
      norm,
      layers.Dense(64, activation='relu'),
      layers.Dense(64, activation='relu'),
      layers.Dense(1)
  ])

  model.compile(loss='mean_absolute_error',
                optimizer=tf.keras.optimizers.Adam(0.001))
  return model

In [ ]:
dnn_cx_model = build_and_compile_model(cx_normalizer)
dnn_cx_model.summary()


In [ ]:
history = dnn_cx_model.fit(
    train_features['cx_mm'],
    train_labels,
    validation_split=0.2,
    verbose=0, epochs=100)

In [ ]:
plot_loss(history)


In [ ]:
x = tf.linspace(-2, 2, 251)
y = dnn_cx_model.predict(x)
plot_cx(x, y)


In [ ]:
test_results['dnn_cx_model'] = dnn_cx_model.evaluate(
    test_features['cx_mm'], test_labels,
    verbose=0)



DNN Multiple inputs

In [ ]:
dnn_model = build_and_compile_model(normalizer)
dnn_model.summary()


In [ ]:
%%time
history = dnn_model.fit(
    train_features,
    train_labels,
    validation_split=0.2,
    verbose=0, epochs=100)


In [ ]:
plot_loss(history)



In [ ]:
test_results['dnn_model'] = dnn_model.evaluate(test_features, test_labels, verbose=0)


In [ ]:
print(test_results)
pd.DataFrame(test_results, index=['Mean absolute error [MPG]']).T
